In [1]:
# paths + core config

from Stage2_expert_stage_utils import *

stage1_checkpoint_path = "./artifacts/stage1_binary_warmstart_best"
expert_all_path = '../Preprocessing+baseline/data/ori_pqal.json'
unlabelled_path = '../Preprocessing+baseline/data/ori_pqau.json'


results_root = "./stage2_cv_results_final"

tokenizer = get_tokenizer()

shared_expert_config = {
    "learning_rate": 8e-6,
    "num_train_epochs": 8,
    "train_batch_size": 8,
    "eval_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "use_weighted_loss": False,
    "use_weighted_sampler": True,
    "runs_root": os.path.join(results_root, "runs"),
    "disable_tqdm": False,
    "logging_strategy": "epoch",
    "logging_steps": None,
    "early_stopping_patience": 2,
    "cv_seed": 42,
    "base_seed": 42,
}

advanced_config = {
    **shared_expert_config,
    "teacher_seed_a": 7,
    "teacher_seed_b": 69,
    "student_base_seed": 420,
    "student_learning_rate": 6e-6,
    "student_num_train_epochs": 8,
    "pseudo_multiplier": 2.50,
    "maybe_boost": 1.25,
    "minimum_pseudo_per_class": 20,
    "min_confidence_by_class": {0: 0.78, 1: 0.62, 2: 0.78},
    "min_margin_by_class": {0: 0.12, 1: 0.05, 2: 0.12},
    "base_pseudo_weight": 0.35,
}

In [2]:
# load datasets, build official split, and build outer folds


expert_all = load_expert_json(expert_all_path)
validate_expert_dataset(expert_all)
expert_all = expert_all.map(add_expert_label)
dev_500, test_500 = build_official_500_500_split(expert_all, seed=42)

unlabelled_dataset = load_unlabelled_json(unlabelled_path)

print_labelled_split_summary(dev_500, test_500, title="500 / 500 SPLIT")

outer_folds = build_outer_cv_folds(
    dev_dataset=dev_500,
    n_splits=10,
    seed=shared_expert_config["cv_seed"],
)

print_outer_fold_summary(outer_folds, title="OUTER 10-FOLD CV WITH 400 TRAIN / 50 CALIBRATION / 50 EVALUATION PER FOLD")

EXPERT DATA CHECK
Examples      : 1000
Label counts  : {'yes': 552, 'no': 338, 'maybe': 110}
Unique labels : ['maybe', 'no', 'yes']


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


500 / 500 SPLIT
dev_500 examples  : 500
dev_500 counts    : {'no': 169, 'maybe': 55, 'yes': 276}

test_500 examples : 500
test_500 counts   : {'no': 169, 'maybe': 55, 'yes': 276}

OUTER 10-FOLD CV WITH 400 TRAIN / 50 CALIBRATION / 50 EVALUATION PER FOLD
fold 1
  train_400       : 400 | {'no': 135, 'maybe': 44, 'yes': 221}
  calibration_50  : 50 | {'no': 17, 'maybe': 6, 'yes': 27}
  eval_50         : 50 | {'no': 17, 'maybe': 5, 'yes': 28}

fold 2
  train_400       : 400 | {'no': 135, 'maybe': 44, 'yes': 221}
  calibration_50  : 50 | {'no': 17, 'maybe': 6, 'yes': 27}
  eval_50         : 50 | {'no': 17, 'maybe': 5, 'yes': 28}

fold 3
  train_400       : 400 | {'no': 135, 'maybe': 44, 'yes': 221}
  calibration_50  : 50 | {'no': 17, 'maybe': 6, 'yes': 27}
  eval_50         : 50 | {'no': 17, 'maybe': 5, 'yes': 28}

fold 4
  train_400       : 400 | {'no': 135, 'maybe': 44, 'yes': 221}
  calibration_50  : 50 | {'no': 17, 'maybe': 6, 'yes': 27}
  eval_50         : 50 | {'no': 17, 'maybe': 5, '

In [3]:
# Pipeline S - Simpler supervised baseline

simple_summary = run_simple_pipeline_cv(
    folds=outer_folds,
    stage1_checkpoint_path=stage1_checkpoint_path,
    tokenizer=tokenizer,
    config=shared_expert_config,
)

print()
print("Simpler pipeline mean outer-fold metrics")
for key, value in simple_summary["mean_outer_metrics"].items():
    print(f"  {key}: {value:.4f}")


PIPELINE S
Strong supervised baseline. For each outer fold we train on 400 expert examples, calibrate on 50 and report performance on the untouched outer 50.



Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Maybe,F1 Yes
1,1.772900,0.635266,0.800000,0.650399,0.909091,0.200000,0.842105
2,1.264200,0.628168,0.780000,0.678531,0.857143,0.363636,0.814815
3,0.813200,0.639780,0.760000,0.685897,0.875000,0.375000,0.807692
4,0.556400,0.678596,0.820000,0.751483,0.857143,0.545455,0.851852
5,0.368500,0.726872,0.800000,0.720625,0.848485,0.461538,0.851852
6,0.229300,0.742051,0.800000,0.720625,0.848485,0.461538,0.851852


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Maybe,F1 Yes
1,1.915800,0.983344,0.460000,0.400919,0.578947,0.100000,0.523810
2,1.360100,0.861404,0.660000,0.486477,0.687500,0.000000,0.771930
3,0.945400,0.873360,0.680000,0.481720,0.645161,0.000000,0.800000
4,0.633800,0.936798,0.700000,0.495833,0.687500,0.000000,0.800000
5,0.294000,1.058081,0.680000,0.481720,0.645161,0.000000,0.800000
6,0.260200,1.135120,0.680000,0.481720,0.645161,0.000000,0.800000


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

fold=10 | macro_f1=0.5023 | acc=0.7000 | f1_no=0.7273 | f1_maybe=0.0000 | f1_yes=0.7797

Simpler pipeline mean outer-fold metrics
  accuracy: 0.7080
  f1_maybe: 0.1922
  f1_no: 0.7423
  f1_yes: 0.7922
  macro_f1: 0.5755


In [ ]:
# Pipeline E - Advanced semi-supervised expert pipeline

advanced_summary = run_advanced_pipeline_cv(
    folds=outer_folds,
    stage1_checkpoint_path=stage1_checkpoint_path,
    unlabelled_dataset=unlabelled_dataset,
    tokenizer=tokenizer,
    config=advanced_config,
)

print()
print("Advanced pipeline mean outer-fold metrics")
for key, value in advanced_summary["mean_outer_metrics"].items():
    print(f"  {key}: {value:.4f}")


PIPELINE E
Advanced semi-supervised pipeline. For each outer fold we train two teachers on 400 expert examples, calibrate them on 50, score the full PQA-U unlabeled pool, select class-aware soft pseudo-labels, train the model, recalibrate it, and report performance on the untouched outer 50.



Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Maybe,F1 Yes
1,2.405300,0.942443,0.540000,0.435556,0.640000,0.000000,0.666667
2,1.568800,0.784057,0.720000,0.524816,0.810811,0.000000,0.763636
3,1.085900,0.693250,0.780000,0.564891,0.875000,0.000000,0.819672
4,0.587200,0.682671,0.820000,0.671842,0.882353,0.285714,0.847458
5,0.410200,0.726759,0.780000,0.631944,0.812500,0.250000,0.833333
6,0.239900,0.842028,0.760000,0.609916,0.774194,0.222222,0.833333


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/61249 [00:00<?, ? examples/s]

Map:   0%|          | 0/61249 [00:00<?, ? examples/s]

Map:   0%|          | 0/1428 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 No,F1 Maybe,F1 Yes
1,0.441900,0.722137,0.760000,0.537967,0.756757,0.000000,0.857143
2,0.347600,0.719033,0.760000,0.544974,0.777778,0.000000,0.857143
3,0.296000,0.718236,0.780000,0.624242,0.800000,0.200000,0.872727
4,0.258200,0.692733,0.780000,0.626455,0.800000,0.222222,0.857143
5,0.235100,0.705723,0.800000,0.642398,0.800000,0.250000,0.877193
6,0.225600,0.707052,0.800000,0.642398,0.800000,0.250000,0.877193


In [ ]:
# Compare both pipelines on the development side and choose winner

comparison = compare_pipeline_summaries(
    simple_summary=simple_summary,
    advanced_summary=advanced_summary,
)

winner = comparison["winner"]
loser = comparison["loser"]

print()
print("winner pipeline")
print(f"  {winner['pipeline_name']}")

save_json(simple_summary, os.path.join(results_root, "simple_summary.json"))
save_json(advanced_summary, os.path.join(results_root, "advanced_summary.json"))
save_json(comparison, os.path.join(results_root, "pipeline_comparison.json"))

In [ ]:
# official test result for the winning pipeline

winning_config = shared_expert_config if winner["pipeline_name"] == "simple_supervised" else advanced_config

winning_test_result = run_final_test_committee(
    pipeline_name=winner["pipeline_name"],
    stage1_checkpoint_path=stage1_checkpoint_path,
    dev_500=dev_500,
    test_500=test_500,
    tokenizer=tokenizer,
    config=winning_config,
    unlabelled_dataset=unlabelled_dataset if winner["pipeline_name"] == "advanced_ssl" else None,
)

save_json(winning_test_result, os.path.join(results_root, "winning_test_result.json"))